# Supervised losses pitch rate control

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [2]:
import ray
import numba as nb
import numpy as np
import xarray.ufuncs as xrf
import matplotlib.pyplot as plt
import pickle

from tqdm.auto import trange
from math import radians, pi, sin
from numba.experimental import jitclass
from IPython.display import display, JSON
import ipywidgets as widgets

from ray import tune
from ray.tune.registry import register_env
from ray.rllib.agents import ppo
from ray.rllib.models import ModelCatalog
from ray.rllib.models.tf.tf_modelv2 import TFModelV2
from ray.rllib.models.tf.fcnet import FullyConnectedNetwork
from ray.tune.utils.log import Verbosity
from ray.tune import JupyterNotebookReporter
# Verbosity.V0_MINIMAL
# Verbosity.V1_EXPERIMENT
# Verbosity.V2_TRIAL_NORM
# Verbosity.V3_TRIAL_DETAILS

from cw.filters import smooth_signal
from cw.vdom import hyr
from traj1.logger import Logger
from environment import LauncherV1SubOrbital, Stage, AP_PITCH_RATE_CONTROL
from reporter import NotebookReporter

Instructions for updating:
non-resource variables are not supported in the long term


## Configure

In [3]:
# tensorboard --logdir=~/ray_results --port=8270 --host=0.0.0.0

In [4]:
register_env("LauncherV1SubOrbital", LauncherV1SubOrbital)

In [5]:
hyr(ray.init(dashboard_host="0.0.0.0"))

2021-05-15 00:30:48,462	INFO services.py:1267 -- View the Ray dashboard at http://0.0.0.0:8265


dict[9]:

node_ip_addressstr[9]:

10.10.3.2

raylet_ip_addressstr[9]:

10.10.3.2

redis_addressstr[14]:

10.10.3.2:6379

object_store_addressstr[70]:

/tmp/ray/session_2021-05-15_00-30-46_926306_15906/sockets/plasma_store

raylet_socket_namestr[64]:

/tmp/ray/session_2021-05-15_00-30-46_926306_15906/sockets/raylet

webui_urlstr[12]:

0.0.0.0:8265

session_dirstr[49]:

/tmp/ray/session_2021-05-15_00-30-46_926306_15906

metrics_export_portint:

34659

node_idstr[56]:

4290c46ca9955698d2486a3ee1d85c5a89f932aada1f24cf51699c24

In [ ]:
ray.shutdown()

In [6]:
class CustomModel(TFModelV2):
    def __init__(self, obs_space, action_space, num_outputs, model_config, name):
        super().__init__(obs_space, action_space, num_outputs, model_config, name)

        self.fcnet = FullyConnectedNetwork(
            self.obs_space,
            self.action_space,
            num_outputs,
            model_config,
            name="fcnet")

    def forward(self, input_dict, state, seq_lens):
        return self.fcnet(input_dict, state, seq_lens)
    
    def value_function(self):
        return self.fcnet.value_function()
    
ModelCatalog.register_custom_model("custom_model", CustomModel)

In [1]:
tune.run(
    ppo.PPOTrainer,
    config={
        "env": "LauncherV1SubOrbital",
        "num_workers": 4,
        "num_gpus": 0,
        "batch_mode": "complete_episodes",
        "env_config": {},
        "model": {
            "custom_model": "custom_model",
            "custom_model_config": {},
        }
    },
    checkpoint_freq=10,
    checkpoint_at_end=True,
    local_dir="./results",
    name="ppo_sl",
    progress_reporter=JupyterNotebookReporter(True),
    verbose=Verbosity.V0_MINIMAL,
    resume=True,
#     restore="",
)

NameError: name 'tune' is not defined